# Option A – v4: fair deep baselines and area boundaries

This notebook is **self-contained**. It needs **no input dataset**: it downloads the benchmark and the official model code itself.

It does two things:

1. **Graph WaveNet and MTGNN trained like our forecaster.** Both models are re-trained with a Poisson likelihood on *all* cell-days and early-stopped on the validation Poisson deviance, over 5 seeds in both cities. Our forecaster was trained this way, whereas in v1/v2 the deep models used ST-HSL's squared-error loss and its crime-day selection rule. The question is whether "gradient-boosted Poisson beats every re-run deep model on all cell-days" survives when the deep models get the same loss and the same selection rule. (≈1–2 h on a T4.)
2. **Area boundaries** for the weekly panels (Chicago community areas, NYPD precincts, San Francisco analysis neighbourhoods) are downloaded from the three open-data portals. They are used for the maps and the spatial-autocorrelation analysis of the paper. (≈1 min.)

**Kaggle settings:** *Accelerator → GPU T4 x2*, *Internet → On*, then *Save Version → Save & Run All (Commit)*.

**Output:** `optionA_outputs_v4.zip`, containing:
- `tables/e01_poisson_retrain_runs.csv`
- `tables/e02_poisson_retrain_summary.csv`
- `tables/e03_boundary_sources.csv`
- `predictions/benchmark_{NYC,CHI}_poisson_predictions.npz`
- `boundaries/*.geojson`
- logs

If the session stops early, add this notebook's partial output as input and run it again. Finished runs are skipped.

## Setup (as in v1; output zip `optionA_outputs_v4.zip`)

In [ ]:
import os, sys, time, json, math, gc, re, warnings, zipfile, shutil, subprocess, random, traceback, hashlib, io, contextlib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
T0 = time.time()

# ------------------------------------------------------------------ configuration
FAST        = os.environ.get("OA_FAST", "0") == "1"          # quick smoke test only
TIME_LIMIT_H = 11.4                                           # stop optional work before Kaggle's 12-hour limit
START_YEAR  = int(os.environ.get("OA_START_YEAR", "2016"))
TRAIN_END   = pd.Timestamp("2023-12-31")                      # same split as the capstone
VAL_END     = pd.Timestamp("2024-06-30")                      # calibration / validation block; test = after this date
CITIES      = ["CHI", "NYC", "SF"]
CITY_NAME   = {"CHI": "Chicago", "NYC": "New York", "SF": "San Francisco"}
DROP_LAST_WEEKS = {"CHI": 1, "NYC": 4, "SF": 2}               # late-reported records make the newest weeks incomplete
HORIZONS    = [1, 2, 3, 4]
NN_HORIZONS = [1, 4]                                          # neural forecasters (slower) are run for these horizons
SEEDS_TREE  = [0] if FAST else [0, 1, 2]
SEEDS_NN    = [0] if FAST else [0, 1, 2]
SEEDS_BENCH = [0] if FAST else [0, 1, 2]                      # ST-HSL, Graph WaveNet, MTGNN, our benchmark forecaster
SEEDS_STSHN = [0] if FAST else [0, 1]                         # ST-SHN (TensorFlow, batch size 1, slower)
ROLL_YEARS  = [2021, 2022] if FAST else [2020, 2021, 2022, 2023, 2024]
ALPHA       = 0.10                                            # 90 % intervals
SEED        = 42

KAGGLE = Path("/kaggle/working").exists()
WORK   = Path("/kaggle/working") if KAGGLE else Path(os.environ.get("OA_WORK", Path.cwd()))
OUT    = WORK / "optionA_outputs_v4"; FIG = OUT / "figures"; TAB = OUT / "tables"; LOGS = OUT / "logs"; PRED = OUT / "predictions"
DATA   = WORK / "raw_data"; REPOS = WORK / "repos"
for p in (OUT, FIG, TAB, LOGS, PRED, DATA, REPOS): p.mkdir(parents=True, exist_ok=True)

def hours_used(): return (time.time() - T0) / 3600
def time_left(): return TIME_LIMIT_H - hours_used()
def log(msg):
    line = f"[{hours_used():5.2f} h] {msg}"; print(line, flush=True)
    with open(LOGS / "progress.log", "a") as f: f.write(line + "\n")
def savefig(name):
    plt.tight_layout(); plt.savefig(FIG / f"{name}.png", dpi=140, bbox_inches="tight"); plt.show(); plt.close("all")
def save_table(df, name, index=False):
    df.to_csv(TAB / f"{name}.csv", index=index); return df
def refresh_zip():
    """Zip everything in the output folder (called after every part)."""
    dest = WORK / "optionA_outputs_v4.zip"; tmp = WORK / "optionA_outputs_v4.zip.part"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for p in OUT.rglob("*"):
            if p.is_file(): z.write(p, "optionA_outputs_v4/" + str(p.relative_to(OUT)))
    tmp.replace(dest); log(f"zip refreshed: {dest} ({dest.stat().st_size / 1e6:.1f} MB)")

STATUS = {}
@contextlib.contextmanager
def part(name):
    """Run one part; on an error write the traceback to logs/errors.log and continue with the next part."""
    t = time.time(); log(f"===== {name} started"); STATUS[name] = "running"
    try:
        yield; STATUS[name] = f"ok ({(time.time() - t) / 60:.0f} min)"
    except Exception:
        tb = traceback.format_exc(); STATUS[name] = "FAILED – see logs/errors.log"
        print(tb); open(LOGS / "errors.log", "a").write(f"===== {name}\n{tb}\n")
    finally:
        log(f"===== {name}: {STATUS[name]}"); json.dump(STATUS, open(LOGS / "status.json", "w"), indent=1)
        try: refresh_zip()
        except Exception as e: print("zip failed:", e)

def pip_install(*pkgs):
    for p in pkgs:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], capture_output=True, timeout=900)
try: import lightgbm as lgb
except ImportError: pip_install("lightgbm"); import lightgbm as lgb
import torch, torch.nn as nn, torch.nn.functional as F
import xgboost as xgb
from scipy.stats import poisson as _pois, nbinom as _nb, kendalltau, spearmanr

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed()

def pick_device():
    if torch.cuda.is_available():
        try: torch.ones(8, device="cuda").sum().item(); return torch.device("cuda")
        except Exception as e: print("CUDA present but unusable with this PyTorch build:", str(e)[:120])
    return torch.device("cpu")
DEVICE = pick_device()
def xgb_device():
    try:
        b = xgb.XGBRegressor(n_estimators=2, device="cuda", tree_method="hist"); b.fit(np.random.rand(64, 3), np.random.rand(64)); return "cuda"
    except Exception: return "cpu"
XGB_DEVICE = xgb_device() if DEVICE.type == "cuda" else "cpu"
VERSIONS = dict(python=sys.version.split()[0], numpy=np.__version__, pandas=pd.__version__, torch=torch.__version__, xgboost=xgb.__version__,
                lightgbm=lgb.__version__, device=str(DEVICE), gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "none", fast=FAST)
json.dump(VERSIONS, open(LOGS / "environment.json", "w"), indent=1)
log(f"environment: {VERSIONS}")
if DEVICE.type == "cpu" and not FAST: print("WARNING: no usable GPU. Switch the accelerator to 'GPU T4 x2', or expect a much longer run.")

# ------------------------------------------------------------------ shared metric helpers
def pdev(y, mu):
    mu = np.maximum(np.asarray(mu, float), 1e-6); y = np.asarray(y, float)
    return 2 * np.mean(np.where(y > 0, y * np.log(np.maximum(y, 1e-12) / mu), 0) - (y - mu))
def unit_dev(y, mu):
    mu = np.maximum(np.asarray(mu, float), 1e-6); y = np.asarray(y, float)
    return 2 * (np.where(y > 0, y * np.log(np.maximum(y, 1e-12) / mu), 0) - (y - mu))
def block_boot(diff, blocks, B=1000, seed=SEED):
    """Mean of `diff` and a 95 % CI from a bootstrap that resamples whole blocks (weeks or days)."""
    wk = pd.factorize(np.asarray(blocks))[0]; s = np.bincount(wk, diff); n = np.bincount(wk); r = np.random.default_rng(seed)
    idx = r.integers(0, len(s), (B, len(s))); est = s[idx].sum(1) / n[idx].sum(1)
    return float(np.mean(diff)), float(np.percentile(est, 2.5)), float(np.percentile(est, 97.5))

# ------------------------------------------------------------------ shared forecasters (gradient-boosted Poisson models)
def fit_lgb_pois(Xtr, ytr, Xes, yes, seed=0, n_est=None):
    m = lgb.LGBMRegressor(objective="poisson", n_estimators=n_est or (150 if FAST else 3000), learning_rate=0.05, num_leaves=127, min_child_samples=20,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=seed, verbose=-1, n_jobs=-1)
    m.fit(Xtr, ytr, eval_set=[(Xes, yes)], callbacks=[lgb.early_stopping(50, verbose=False)]); return m
def fit_xgb_pois(Xtr, ytr, Xes, yes, seed=0):
    m = xgb.XGBRegressor(objective="count:poisson", n_estimators=150 if FAST else 2000, learning_rate=0.05, max_depth=7, subsample=0.8,
                         colsample_bytree=0.8, min_child_weight=5, tree_method="hist", device=XGB_DEVICE, enable_categorical=True,
                         max_cat_to_onehot=1, random_state=seed, early_stopping_rounds=50, eval_metric="poisson-nloglik")
    m.fit(Xtr, ytr, eval_set=[(Xes, yes)], verbose=False); return m
def fit_lgb_quantile(Xtr, ytr, Xes, yes, q, seed=0):
    m = lgb.LGBMRegressor(objective="quantile", alpha=q, n_estimators=150 if FAST else 2000, learning_rate=0.05, num_leaves=63, min_child_samples=50,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=seed, verbose=-1, n_jobs=-1)
    m.fit(Xtr, ytr, eval_set=[(Xes, yes)], callbacks=[lgb.early_stopping(50, verbose=False)]); return m
def fit_lgb_l2(Xtr, ytr, Xes, yes, seed=0):
    m = lgb.LGBMRegressor(objective="l2", n_estimators=150 if FAST else 2000, learning_rate=0.05, num_leaves=31, min_child_samples=100,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=seed, verbose=-1, n_jobs=-1)
    m.fit(Xtr, ytr, eval_set=[(Xes, yes)], callbacks=[lgb.early_stopping(50, verbose=False)]); return m

# ------------------------------------------------------------------ shared interval helpers (all calibrated on a held-out block only)
def conformal_q(scores, alpha=ALPHA):
    scores = np.asarray(scores, float); n = len(scores)
    if n == 0: return np.inf
    lev = np.ceil((n + 1) * (1 - alpha)) / n
    return np.inf if lev > 1 else float(np.quantile(scores, lev, method="higher"))
def norm_score(y, mu): return np.abs(y - mu) / np.sqrt(mu + 0.5)
def int_from_q(mu, q, scale=None):
    s = np.sqrt(mu + 0.5) if scale is None else scale
    lo = np.maximum(np.ceil(mu - q * s), 0); hi = np.floor(mu + q * s)
    bad = hi < lo; r = np.round(mu); return np.where(bad, r, lo), np.where(bad, r, hi)
def mondrian_edges(mu_cal, k=5): return np.unique(np.quantile(mu_cal, np.linspace(0, 1, k + 1)[1:-1]))
def nb_size(mu, y):                                            # negative-binomial size by moments: Var = mu + mu^2 / r
    return float(min(max(np.sum(mu ** 2) / max(np.sum((y - mu) ** 2 - mu), 1e-6), 1e-3), 1e4))
def parametric_interval(mu, kind, r=None):
    mu = np.maximum(mu, 1e-6)
    if kind == "Poisson": return _pois.ppf(ALPHA / 2, mu), _pois.ppf(1 - ALPHA / 2, mu)
    p = r / (r + mu); return _nb.ppf(ALPHA / 2, r, p), _nb.ppf(1 - ALPHA / 2, r, p)
def interval_score(y, lo, hi): return (hi - lo) + (2 / ALPHA) * np.maximum(lo - y, 0) + (2 / ALPHA) * np.maximum(y - hi, 0)
def coverage_report(y, lo, hi, vq=None, area=None):
    """Overall coverage, width, interval score, coverage per volume quintile and per area."""
    cov = (y >= lo) & (y <= hi); out = dict(coverage=cov.mean(), width=(hi - lo).mean(), interval_score=interval_score(y, lo, hi).mean())
    if vq is not None:
        for q in range(1, 6): out[f"cov_Q{q}"] = cov[vq == q].mean() if (vq == q).any() else np.nan
        out["max_abs_dev_from_90"] = np.nanmax([abs(out[f"cov_Q{q}"] - (1 - ALPHA)) for q in range(1, 6)])
        out["gap_Q5_minus_Q1"] = out["cov_Q5"] - out["cov_Q1"]
    if area is not None:
        ac = pd.Series(cov).groupby(np.asarray(area)).mean()
        out["areas_within_5pts"] = float((ac.sub(1 - ALPHA).abs() <= 0.05).mean()); out["worst_area_dev"] = float(ac.sub(1 - ALPHA).abs().max())
        out["sd_area_coverage"] = float(ac.std())
    return out

In [ ]:
# ------------------------------------------------------------------ v4 configuration
SEEDS_V4 = [0] if FAST else [0, 1, 2, 3, 4]
V4_MODELS = ["Graph WaveNet", "MTGNN"]
BENCH_CITIES = ["NYC", "CHI"]
BENCH_CATS = {"NYC": ["Burglary", "Larceny", "Robbery", "Assault"], "CHI": ["Theft", "Battery", "Assault", "Damage"]}
# the commits used in v1/v2 (table b01); v4 checks out exactly these
PINNED = {"STHSL": ("https://github.com/LZH-YS1998/STHSL", "6f3429c07f88ada44b5c34a0a098e95eedf9cbd6"),
          "Graph-WaveNet": ("https://github.com/nnzhan/Graph-WaveNet", "6b162e80c59a1d494809252eca055cff93dc66b1"),
          "MTGNN": ("https://github.com/nnzhan/MTGNN", "f811746fa7022ebf336f9ecd2434af5f365ecbf6")}
RUNS = PRED / "poisson_runs"; RUNS.mkdir(parents=True, exist_ok=True)
BOUND = OUT / "boundaries"; BOUND.mkdir(parents=True, exist_ok=True)

# resume: copy finished runs from an earlier (partial) v4 output added as input
INPUT_ROOT = Path(os.environ.get("OA_INPUT", "/kaggle/input"))
if INPUT_ROOT.exists():
    for f in INPUT_ROOT.rglob("poisson_runs/*.npz"):
        if f.is_file() and not (RUNS / f.name).exists():
            shutil.copy(f, RUNS / f.name)
            if f.with_suffix(".json").exists(): shutil.copy(f.with_suffix(".json"), RUNS / f.with_suffix(".json").name)
    for f in INPUT_ROOT.rglob("poisson_runs/*.npz"):
        if f.is_dir():                                                   # Kaggle unpacks .npz files of uploaded zips into folders
            dst = RUNS / f.name
            if not dst.exists(): np.savez_compressed(dst, **{p.stem: np.load(p) for p in f.glob("*.npy")})
log(f"v4: seeds {SEEDS_V4}; finished runs found: {len(list(RUNS.glob('*.npz')))}")

def git_clone_pinned(name):
    url, commit = PINNED[name]; dest = REPOS / name
    if not dest.exists():
        r = subprocess.run(["git", "clone", url, str(dest)], capture_output=True, text=True)
        if r.returncode != 0: raise RuntimeError(f"could not clone {name}: {r.stderr[-300:]}")
    r = subprocess.run(["git", "checkout", "-q", commit], cwd=dest, capture_output=True, text=True)
    head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=dest, capture_output=True, text=True).stdout.strip()
    return dict(repository=name, url=url, commit_v1=commit, commit_v4=head, identical=(head == commit))

def bench_load(c):
    import pickle
    t = {s: np.asarray(pickle.load(open(REPOS / "STHSL" / "Datasets" / f"{c}_crime" / f"{s}.pkl", "rb")), np.float32) for s in ("trn", "val", "tst")}
    return np.concatenate([t["trn"], t["val"], t["tst"]], 2), t["trn"].shape[2], t["val"].shape[2], t["tst"].shape[2]

with part("v4.0 official code at the v1 commits"):
    VERS = save_table(pd.DataFrame([git_clone_pinned(n) for n in PINNED]), "e00_code_versions_v4"); display(VERS)

## Part 1. Area boundaries of the weekly panels
Downloaded from the three open-data portals (known dataset ids first, then the portal's catalogue search). A file is kept only if its area key matches at least 90 % of the panel's areas.

In [ ]:
# ---------------------------------------------------------------- area boundaries of the weekly panels (for maps and Moran's I)
import requests
EXPECTED = {
    "CHI": [str(i) for i in range(1, 78)],
    "NYC": "1 5 6 7 9 10 13 14 17 18 19 20 22 23 24 25 26 28 30 32 33 34 40 41 42 43 44 45 46 47 48 49 50 52 60 61 62 63 66 67 68 69 70 71 72 73 75 76 77 78 79 81 83 84 88 90 94 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 120 121 122 123".split(),
    "SF": ["Bayview Hunters Point", "Bernal Heights", "Castro/Upper Market", "Chinatown", "Excelsior", "Financial District/South Beach", "Glen Park",
           "Golden Gate Park", "Haight Ashbury", "Hayes Valley", "Inner Richmond", "Inner Sunset", "Japantown", "Lakeshore", "Lincoln Park",
           "Lone Mountain/USF", "Marina", "McLaren Park", "Mission", "Mission Bay", "Nob Hill", "Noe Valley", "North Beach", "Oceanview/Merced/Ingleside",
           "Outer Mission", "Outer Richmond", "Pacific Heights", "Portola", "Potrero Hill", "Presidio", "Presidio Heights", "Russian Hill", "Seacliff",
           "South of Market", "Sunset/Parkside", "Tenderloin", "Treasure Island", "Twin Peaks", "Visitacion Valley", "West of Twin Peaks", "Western Addition"]}
BSRC = {"CHI": dict(domain="data.cityofchicago.org", ids=["igwz-8jzy", "cauf-n9fc"], q="community areas boundaries"),
        "NYC": dict(domain="data.cityofnewyork.us", ids=["y76i-bdw7", "78dh-3ptz", "kmub-vria"], q="police precincts"),
        "SF":  dict(domain="data.sfgov.org", ids=["j2bu-swwd", "p5b7-5n3h", "xfcw-9evu"], q="analysis neighborhoods")}
def norm(v):
    s = str(v).strip()
    try: return str(int(float(s)))
    except ValueError: return s.lower()
def try_geojson(domain, did):
    for url in (f"https://{domain}/api/geospatial/{did}?method=export&format=GeoJSON", f"https://{domain}/resource/{did}.geojson?$limit=5000"):
        try:
            r = requests.get(url, timeout=60)
            if r.status_code == 200 and r.text.lstrip().startswith("{"):
                g = r.json()
                if g.get("features"): return g, url
        except Exception as e: log(f"    {url}: {str(e)[:120]}")
    return None, None
def match_key(g, expected):
    exp = {norm(e) for e in expected}; best = (None, 0)
    for key in g["features"][0].get("properties", {}):
        vals = {norm(f["properties"].get(key)) for f in g["features"]}
        hit = len(exp & vals)
        if hit > best[1]: best = (key, hit)
    return best
def catalog_ids(domain, q):
    try:
        r = requests.get("https://api.us.socrata.com/api/catalog/v1", params=dict(domains=domain, q=q, limit=15), timeout=60).json()
        return [x["resource"]["id"] for x in r.get("results", [])]
    except Exception as e: log(f"    catalog search failed: {str(e)[:120]}"); return []

with part("v4.3 area boundaries"):
    BROWS = []
    for c, src in BSRC.items():
        done = False
        for did in src["ids"] + [i for i in catalog_ids(src["domain"], src["q"]) if i not in src["ids"]]:
            g, url = try_geojson(src["domain"], did)
            if g is None: continue
            key, hit = match_key(g, EXPECTED[c])
            log(f"  {c} {did}: {len(g['features'])} features, key '{key}' matches {hit}/{len(EXPECTED[c])} areas")
            if hit >= 0.9 * len(EXPECTED[c]):
                json.dump(g, open(BOUND / f"{c}_areas.geojson", "w"))
                BROWS.append(dict(city=c, dataset_id=did, url=url, features=len(g["features"]), key=key, areas_matched=hit, areas_expected=len(EXPECTED[c]),
                                  unmatched=";".join(sorted({norm(e) for e in EXPECTED[c]} - {norm(f["properties"].get(key)) for f in g["features"]}))))
                done = True; break
        if not done: BROWS.append(dict(city=c, dataset_id="NOT FOUND", areas_expected=len(EXPECTED[c])))
    BROWS = save_table(pd.DataFrame(BROWS), "e03_boundary_sources"); display(BROWS)
    missing = BROWS.loc[BROWS.dataset_id == "NOT FOUND", "city"].tolist()
    if missing: log(f"WARNING: no boundary file for {missing} – is Internet switched on (Settings → Internet → On)?")

## Part 2. Graph WaveNet and MTGNN with a Poisson likelihood on all cell-days
Official model code (same patch and hyper-parameters as v1/v2). Only the training loss and the epoch-selection rule change, to the ones our gradient-boosted forecaster uses.

In [ ]:
def load_module(path, name, extra_path=None):
    import importlib.util
    if extra_path and str(extra_path) not in sys.path: sys.path.insert(0, str(extra_path))
    spec = importlib.util.spec_from_file_location(name, path); m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
def grid_adj(R, Cc):
    N = R * Cc; A = np.eye(N, dtype=np.float32)
    for i in range(R):
        for j in range(Cc):
            for di in (-1, 0, 1):
                for dj in (-1, 0, 1):
                    if 0 <= i + di < R and 0 <= j + dj < Cc: A[i * Cc + j, (i + di) * Cc + (j + dj)] = 1
    return A
def make_graph_model(kind, N, K, R, Cc):
    if kind == "Graph WaveNet":
        gd = REPOS / "Graph-WaveNet"; mp = gd / "model_patched.py"
        if not mp.exists(): mp.write_text((gd / "model.py").read_text().replace("nn.Conv1d(", "nn.Conv2d("))
        gw = load_module(mp, "gwn_model_patched")
        A = grid_adj(R, Cc); P = A / A.sum(1, keepdims=True)
        sup = [torch.tensor(P, device=DEVICE), torch.tensor(P.T.copy(), device=DEVICE)]
        return gw.gwnet(DEVICE, N, dropout=0.3, supports=sup, gcn_bool=True, addaptadj=True, aptinit=None, in_dim=K, out_dim=K,
                        residual_channels=32, dilation_channels=32, skip_channels=256, end_channels=512).to(DEVICE)
    mt = load_module(REPOS / "MTGNN" / "net.py", "mtgnn_net", REPOS / "MTGNN")
    return mt.gtnet(True, True, 2, N, DEVICE, predefined_A=None, static_feat=None, dropout=0.3, subgraph_size=min(20, N), node_dim=40,
                    dilation_exponential=1, conv_channels=32, residual_channels=32, skip_channels=64, end_channels=128, seq_length=30,
                    in_dim=K, out_dim=K, layers=3, propalpha=0.05, tanhalpha=3, layer_norm_affline=True).to(DEVICE)

In [ ]:
# ---------------------------------------------------------------- Graph WaveNet / MTGNN with a Poisson likelihood on all cell-days
def train_graph_poisson(kind, c, seed, max_epochs=2 if FAST else 100, patience=10, batch=16, L=30):
    """Same architecture, input window, optimiser and early-stopping patience as v1, but
    loss = Poisson negative log-likelihood on every cell-day (the model outputs a log-rate), and
    the epoch is chosen by the Poisson deviance on all validation cell-days (the rule our forecaster uses)."""
    set_seed(seed); Xall, ntr, nva, nte = bench_load(c); R, Cc, T, K = Xall.shape; N = R * Cc
    S = Xall.reshape(N, T, K); mean, std = float(Xall[:, :, :ntr].mean()), float(Xall[:, :, :ntr].std())
    Z = torch.tensor((S - mean) / std, device=DEVICE); Y = torch.tensor(S, device=DEVICE)
    model = make_graph_model(kind, N, K, R, Cc); opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    back = torch.arange(L, 0, -1, device=DEVICE); offset = math.log(max(mean, 1e-3))     # start at the training mean rate
    def fwd(days):
        win = days[:, None] - back[None]
        x = Z[:, win, :].permute(1, 3, 0, 2)                                    # (B, K, N, L)
        out = model(x)[..., -1]                                                 # (B, K, N)
        return torch.clamp(out.permute(0, 2, 1) + offset, -12.0, 8.0)          # (B, N, K) log-rates
    def predict(days):
        model.eval(); out = []
        with torch.no_grad():
            for s in range(0, len(days), 32): out.append(torch.exp(fwd(days[s:s + 32])).cpu().numpy())
        return np.concatenate(out)
    trd = torch.arange(L, ntr, device=DEVICE); vad = torch.arange(ntr, ntr + nva, device=DEVICE); ted = torch.arange(ntr + nva, T, device=DEVICE)
    yv = S[:, ntr:ntr + nva].transpose(1, 0, 2); best, best_state, bad, best_ep, t, hist = np.inf, None, 0, 0, time.time(), []
    for ep in range(max_epochs):
        model.train(); perm = trd[torch.randperm(len(trd), device=DEVICE)]
        for s in range(0, len(perm), batch):
            b = perm[s:s + batch]
            loss = F.poisson_nll_loss(fwd(b), Y[:, b, :].permute(1, 0, 2), log_input=True, full=False)
            opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
        pv = predict(vad); crit = pdev(yv, pv); hist.append(float(crit))       # validation deviance on all cell-days
        if crit < best - 1e-6: best, bad, best_state, best_ep = crit, 0, {k: v.detach().clone() for k, v in model.state_dict().items()}, ep + 1
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    info = dict(model=kind, city=c, seed=seed, loss="Poisson NLL, all cell-days", selection="validation Poisson deviance, all cell-days",
                epochs_run=ep + 1, epochs_planned=max_epochs, selected_epoch=best_ep, val_deviance=float(best),
                minutes=(time.time() - t) / 60, finished=True, val_history=hist)
    log(f"  {kind} (Poisson) {c} seed {seed}: {ep + 1} epochs, best {best_ep}, val deviance {best:.4f}, {info['minutes']:.1f} min")
    return predict(ted), predict(vad), info

def safe(name): return re.sub(r"[^A-Za-z0-9]+", "_", name)
def run_file(kind, c, s): return RUNS / f"{safe(kind)}_poisson_{c}_s{s}.npz"

with part("v4.1 Graph WaveNet and MTGNN with Poisson loss on all cell-days"):
    JOBS = [(k, c, s) for s in SEEDS_V4 for k in V4_MODELS for c in BENCH_CITIES]   # seed-major: a time-out still leaves every model/city with runs
    for kind, c, s in JOBS:
        f = run_file(kind, c, s)
        if f.exists(): log(f"  {kind} {c} seed {s}: already done"); continue
        if time_left() < 0.4: log(f"  time budget reached before {kind} {c} seed {s} – run the notebook again to finish"); break
        try:
            pt, pv, info = train_graph_poisson(kind, c, s)
            np.savez_compressed(f, pt=pt.reshape(-1).astype(np.float32), pv=pv.reshape(-1).astype(np.float32))
            json.dump(info, open(f.with_suffix(".json"), "w"), default=str)
        except Exception as e:
            log(f"  {kind} {c} seed {s} failed: {str(e)[:300]}"); open(LOGS / "errors.log", "a").write(f"{kind} {c} {s}\n{traceback.format_exc()}\n")
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
        refresh_zip()

## Part 3. Scores under the official (P1), full (P3) and decision (P4) protocols

In [ ]:
# ---------------------------------------------------------------- score the Poisson-trained runs under P1, P3 and P4
def selected_weights(p, k):
    v = np.sort(p)[::-1][k - 1]; above = p > v; tie = p == v
    w = above.astype(float)
    if tie.any(): w[tie] = (k - above.sum()) / tie.sum()
    return w
def hit_rate(p, y, D, N, K, share=0.10):
    P, Y = p.reshape(D, N, K), y.reshape(D, N, K); k = int(round(share * N)); cap = tot = 0.0
    for kk in range(K):
        for d in range(D):
            cap += (selected_weights(P[d, :, kk], k) * Y[d, :, kk]).sum(); tot += Y[d, :, kk].sum()
    return cap / tot, k / N

with part("v4.2 scores under P1, P3 and P4"):
    ROWS = []
    for c in BENCH_CITIES:
        Xall, ntr, nva, nte = bench_load(c); R, Cc, T, K = Xall.shape; N = R * Cc
        y = Xall.reshape(N, T, K)[:, ntr + nva:].transpose(1, 0, 2).reshape(-1); cat = np.tile(np.arange(K), nte * N)
        store = {}
        for kind in V4_MODELS:
            for s in SEEDS_V4:
                f = run_file(kind, c, s)
                if not f.exists(): continue
                p = np.load(f)["pt"].astype(float); info = json.load(open(f.with_suffix(".json")))
                store.setdefault(safe(kind + " (Poisson loss, all cell-days)"), []).append(p)
                pos = y > 0; hr, share = hit_rate(p, y, nte, N, K)
                row = dict(city=c, model=kind, training="Poisson NLL on all cell-days, validation deviance", seed=s,
                           selected_epoch=info["selected_epoch"], epochs_run=info["epochs_run"], minutes=info["minutes"],
                           P1_MAE=np.abs(p - y)[pos].mean(), P1_MAPE=(np.abs(p - y)[pos] / y[pos]).mean(),
                           P3_deviance_all=pdev(y, np.maximum(p, 0.01)), P3_MAE_all=np.abs(p - y).mean(),
                           MAE_zero_days=np.abs(p - y)[~pos].mean(), MAE_crime_days=np.abs(p - y)[pos].mean(),
                           P4_hit_rate=hr, P4_PAI=hr / share)
                for k in range(K):
                    m = pos & (cat == k); row[f"P1_MAE_{BENCH_CATS[c][k]}"] = np.abs(p - y)[m].mean()
                    row[f"P3_deviance_{BENCH_CATS[c][k]}"] = pdev(y[cat == k], np.maximum(p[cat == k], 0.01))
                ROWS.append(row)
        if store:
            out = PRED / f"benchmark_{c}_poisson_predictions.npz"
            if out.is_dir(): shutil.rmtree(out)
            np.savez_compressed(out, y=y, cat=cat, **{k: np.stack(v) for k, v in store.items()})
    E1 = save_table(pd.DataFrame(ROWS), "e01_poisson_retrain_runs")
    if len(E1):
        cols = ["P1_MAE", "P1_MAPE", "P3_deviance_all", "P3_MAE_all", "P4_hit_rate", "P4_PAI", "selected_epoch"]
        E2 = E1.groupby(["city", "model"])[cols].agg(["mean", "std", "min", "max", "count"])
        E2.columns = [f"{a}_{b}" for a, b in E2.columns]; E2 = save_table(E2.reset_index(), "e02_poisson_retrain_summary")
        display(E2[["city", "model"] + [f"{c}_mean" for c in cols] + ["P3_deviance_all_std", "P3_deviance_all_count"]].round(4))
        log("For comparison (v2, 5 seeds, full protocol P3 deviance): ours 0.349 NYC / 0.501 CHI; MTGNN (MSE) 0.361 / 0.515.")